In [2]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import tensorflow as tf

# Load processed train-test datasets
X_train = np.load("X_train.npy")
y_train = np.load("y_train.npy")
X_test = np.load("X_test.npy")
y_test = np.load("y_test.npy")

# Load trained BiLSTM Model
bilstm_model = tf.keras.models.load_model("bilstm_alarm_model.h5")

# **Pastikan input memiliki format (samples, timesteps, features)**
X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

# Extract LSTM Features for XGBoost
lstm_features_train = bilstm_model.predict(X_train)  # Sekarang dalam format (samples, 32)
lstm_features_test = bilstm_model.predict(X_test)

# Train XGBoost Classifier
xgb_model = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, objective='multi:softmax', num_class=4)
xgb_model.fit(lstm_features_train, y_train)

# Predict with XGBoost
y_pred_xgb = xgb_model.predict(lstm_features_test)
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
print(f"XGBoost Model Accuracy (Using LSTM Features): {accuracy_xgb * 100:.2f}%")
print("Classification Report (XGBoost):\n", classification_report(y_test, y_pred_xgb))

# Save XGBoost Model
xgb_model.save_model("xgboost_lstm_model.json")
print("XGBoost model saved as xgboost_lstm_model.json")


2025-05-23 15:03:36.211167: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


4779/4779 ━━━━━━━━━━━━━━━━━━━━ 4s 839us/step
612/612 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  
XGBoost Model Accuracy (Using LSTM Features): 94.81%
Classification Report (XGBoost):
               precision    recall  f1-score   support

           0       0.99      0.91      0.95      9558
           1       0.83      0.96      0.89      1145
           2       0.78      0.99      0.87      2302
           3       0.99      0.99      0.99      6557

    accuracy                           0.95     19562
   macro avg       0.90      0.96      0.92     19562
weighted avg       0.96      0.95      0.95     19562

XGBoost model saved as xgboost_lstm_model.json


In [1]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np
import tensorflow as tf

# Load Data
X_train = np.load("X_train.npy")
y_train = np.load("y_train.npy")
X_test = np.load("X_test.npy")
y_test = np.load("y_test.npy")

# Load BiLSTM Model
bilstm_model = tf.keras.models.load_model("bilstm_alarm_model.h5")

# Reshape for LSTM (samples, timesteps, features)
X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

# Extract Features
lstm_features_train = bilstm_model.predict(X_train)
lstm_features_test = bilstm_model.predict(X_test)

# Load Trained XGBoost Model
xgb_model = xgb.XGBClassifier()
xgb_model.load_model("xgboost_lstm_model.json")

# Predict
y_pred = xgb_model.predict(lstm_features_test)
acc = accuracy_score(y_test, y_pred)

# Display Accuracy and Classification Report
print(f"✅ XGBoost + LSTM Accuracy: {acc:.4f}")
print("\n📊 Classification Report:\n", classification_report(y_test, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\n📉 Confusion Matrix:")
print(cm)

# Hitung FP dan FN per class
print("\n📌 False Positives (FP) dan False Negatives (FN) per class:")
num_classes = cm.shape[0]
for i in range(num_classes):
    FN = np.sum(cm[i, :]) - cm[i, i]
    FP = np.sum(cm[:, i]) - cm[i, i]
    print(f"Class {i}: FP={FP}, FN={FN}")


2025-05-24 09:13:24.593132: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-24 09:13:24.595896: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-24 09:13:24.605112: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748052804.620646  139450 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748052804.625333  139450 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-24 09:13:24.641446: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU ins

4779/4779 ━━━━━━━━━━━━━━━━━━━━ 5s 966us/step 
612/612 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step  
✅ XGBoost + LSTM Accuracy: 0.9481

📊 Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.91      0.95      9558
           1       0.83      0.96      0.89      1145
           2       0.78      0.99      0.87      2302
           3       0.99      0.99      0.99      6557

    accuracy                           0.95     19562
   macro avg       0.90      0.96      0.92     19562
weighted avg       0.96      0.95      0.95     19562


📉 Confusion Matrix:
[[8704  158  619   77]
 [  29 1094   14    8]
 [   9   11 2282    0]
 [  28   62    1 6466]]

📌 False Positives (FP) dan False Negatives (FN) per class:
Class 0: FP=66, FN=854
Class 1: FP=231, FN=51
Class 2: FP=634, FN=20
Class 3: FP=85, FN=91
